# Day 6 · Version B: one model call with the right context

**First-principles question:** What does the model need to see to answer like a trained support agent?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import json
import pandas as pd
from lab.acme import (INTENTS, INTENT_KB, customer_card, kb_by_id, load_cases, load_customers,
                      load_results, save_results)
from lab.evals import score, summary
from lab.llm import chat

cases = load_cases()
customers = load_customers()
kb = kb_by_id()
POLICY = (ROOT / "data" / "acme" / "policy.md").read_text(encoding="utf-8")
print(len(cases), "cases ·", len(kb), "help articles")

## 1 · The system prompt
Who the model is, what it must do, and the rules.

In [ ]:
# TODO: write the instructions in your own words, then keep the policy below them
SYSTEM = f"""You are ...

{POLICY}
"""
print(SYSTEM[:500])

## 2 · The user message
Today the help articles are picked by the *expected* intent: an 'oracle', the best possible search. Project 2 replaces this with real search and measures the difference.

In [ ]:
def user_message(case):
    customer = customers[case["customer_id"]]
    articles = [kb[a] for a in INTENT_KB[case["expected_intent"]]]     # oracle context (for now)
    # TODO: return one string with three parts:
    #   1. customer_card(customer)
    #   2. the articles, each as "KBxx · title: text"
    #   3. the customer's email: case["message"]
    return ""

print(user_message(cases[0]))

## 3 · Ask for JSON
A JSON schema forces the shape of the answer.

In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {
        "intent": {"type": "string", "enum": INTENTS},
        "reply": {"type": "string"},
        "needs_human": {"type": "boolean"},
    },
    "required": ["intent", "reply", "needs_human"],
}

def version_b(case, temperature=0.0, run=0):
    r = chat([{"role": "system", "content": SYSTEM},
              {"role": "user", "content": user_message(case)}],
             schema=SCHEMA, temperature=temperature, run=run)
    stats = {"tokens_in": r["tokens_in"], "tokens_out": r["tokens_out"], "seconds": r["seconds"]}
    # TODO: turn r["content"] into a dict with json.loads and return it merged with stats.
    #       If parsing fails (json.JSONDecodeError), return
    #       {"intent": "other", "reply": "", "needs_human": True, "parse_error": True} merged with stats.
    ...

version_b(cases[0])

## 4 · Every case, scored, next to version A
Answers are cached in `data/cache/`: re-running is free.

In [ ]:
by_id = {c["id"]: c for c in cases}
b_rows = [dict(version_b(c), id=c["id"]) for c in cases]
save_results("B", b_rows)
a_rows = load_results("A")

table = pd.DataFrame({
    "A": summary([score(by_id[r["id"]], r) for r in a_rows]),
    "B": summary([score(by_id[r["id"]], r) for r in b_rows]),
}).map("{:.0%}".format)
print("JSON parse failures:", sum(bool(r.get("parse_error")) for r in b_rows))
table

## 5 · Wins and losses

In [ ]:
a_pass = {r["id"]: score(by_id[r["id"]], r)["passed"] for r in a_rows}
b_pass = {r["id"]: score(by_id[r["id"]], r)["passed"] for r in b_rows}
print("B won: ", [i for i in b_pass if b_pass[i] and not a_pass.get(i)])
print("B lost:", [i for i in b_pass if not b_pass[i] and a_pass.get(i)])

**3 wins and 3 losses, one line each:**

- …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert "You are ..." not in SYSTEM, "Write your own system prompt in step 1"
msg = user_message(cases[0])
assert cases[0]["message"] in msg, "user_message() must include the email"
assert customers[cases[0]["customer_id"]]["name"].split()[0] in msg, "...and the customer card"
assert len(b_rows) == len(cases) and all("intent" in r for r in b_rows)
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")